# Fine-tune Whisper on Tamil → push to Hugging Face (Venky0411)

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. No edits needed.

Default data = **Common Voice Tamil** (`abar-uwc/tamil-common-voice_v21`, Parquet — no
loading script, works on the latest `datasets`). Set `USE_FLEURS = True` for a smaller set.

> ⚠️ Contains a write token. Don't share/commit it publicly; rotate it at
> hf.co/settings/tokens after the run.

In [ ]:
!pip -q install --upgrade "transformers>=4.44" "datasets[audio]>=2.20" "accelerate>=0.33" evaluate jiwer librosa soundfile huggingface_hub

In [ ]:
# ===== CONFIG (already set for you) =====
USE_FLEURS = False   # False = Common Voice Tamil (45k clips). True = FLEURS (small & fast).

MODEL_NAME    = "openai/whisper-small"   # whisper-base / whisper-tiny = faster, lower quality
LANGUAGE      = "tamil"
HF_USERNAME   = "Venky0411"
# HF_TOKEN is fetched securely in the login cell (Colab Secret / env var / prompt) — never hard-code
HUB_REPO      = f"{HF_USERNAME}/whisper-small-ta-saaras"

if USE_FLEURS:
    # parquet-native, has train/test splits
    DATASET, CONFIG, TEXT_COL, HAS_SPEAKER, NEEDS_SPLIT = "google/fleurs", "ta_in", "transcription", False, False
else:
    # parquet-native Common Voice Tamil; single 'train' split -> we carve a test set
    DATASET, CONFIG, TEXT_COL, HAS_SPEAKER, NEEDS_SPLIT = "abar-uwc/tamil-common-voice_v21", None, "sentence", True, True

MAX_TRAIN_SAMPLES = 8000    # cap for a weekend T4 run; set None for the full split
MAX_EVAL_SAMPLES  = 1000
MAX_STEPS         = 2000    # ~2-3h on T4 for whisper-small
BATCH_SIZE        = 16      # lower to 8 if you hit CUDA OOM
GRAD_ACCUM        = 1
LEARNING_RATE     = 1e-5
print(f"data={DATASET}  ->  push to {HUB_REPO}")

In [ ]:
from huggingface_hub import login
import os
try:
    from google.colab import userdata           # add a Colab Secret named HF_TOKEN (🔑 panel)
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    from getpass import getpass
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass('HF write token: ')
login(token=HF_TOKEN)

In [ ]:
from datasets import load_dataset, Audio

def _load(split):
    return load_dataset(DATASET, CONFIG, split=split) if CONFIG else load_dataset(DATASET, split=split)

if NEEDS_SPLIT:
    full = _load("train")
    parts = full.train_test_split(test_size=0.05, seed=42)
    train, test = parts["train"], parts["test"]
else:
    train, test = _load("train"), _load("test")

keep = {"audio", TEXT_COL} | ({"client_id"} if HAS_SPEAKER else set())
train = train.remove_columns([c for c in train.column_names if c not in keep])
test  = test.remove_columns([c for c in test.column_names if c not in keep])
print(train, test)

In [ ]:
# Speaker-independent eval (Common Voice only) + size caps
if HAS_SPEAKER:
    train_spk = set(train.unique("client_id"))
    test = test.filter(lambda x: x["client_id"] not in train_spk)
    print("speaker-independent test rows:", len(test))

train = train.shuffle(seed=42)
if MAX_TRAIN_SAMPLES:
    train = train.select(range(min(MAX_TRAIN_SAMPLES, len(train))))
if MAX_EVAL_SAMPLES:
    test = test.select(range(min(MAX_EVAL_SAMPLES, len(test))))
print("using", len(train), "train /", len(test), "test")

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task="transcribe")
train = train.cast_column("audio", Audio(sampling_rate=16000))
test  = test.cast_column("audio", Audio(sampling_rate=16000))

def prepare(batch):
    a = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        a["array"], sampling_rate=16000).input_features[0]
    batch["labels"] = processor.tokenizer(batch[TEXT_COL]).input_ids
    return batch

train = train.map(prepare, remove_columns=train.column_names, num_proc=2)
test  = test.map(prepare,  remove_columns=test.column_names,  num_proc=2)

# Whisper's decoder caps at 448 tokens; drop rows with over-long transcripts (bad data)
MAX_LABEL_LENGTH = 448
train = train.filter(lambda x: len(x["labels"]) <= MAX_LABEL_LENGTH)
test  = test.filter(lambda x: len(x["labels"]) <= MAX_LABEL_LENGTH)
print("after length filter:", len(train), "train /", len(test), "test")

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

In [ ]:
import evaluate
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {
        "wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": 100 * cer_metric.compute(predictions=pred_str, references=label_str),
    }

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language = LANGUAGE
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

args = Seq2SeqTrainingArguments(
    output_dir="./whisper-ta",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=200,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=500,
    logging_steps=25,
    report_to=["none"],
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    push_to_hub=True,
    hub_model_id=HUB_REPO,
)

trainer = Seq2SeqTrainer(
    args=args, model=model, train_dataset=train, eval_dataset=test,
    data_collator=data_collator, compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

In [ ]:
trainer.train()

In [ ]:
print(trainer.evaluate())

In [ ]:
processor.save_pretrained("./whisper-ta")
trainer.push_to_hub(**{
    "dataset": DATASET, "language": "ta",
    "model_name": "Whisper Small Tamil (Saaras-style)",
    "finetuned_from": MODEL_NAME, "tasks": "automatic-speech-recognition",
})
processor.push_to_hub(HUB_REPO)
print("Pushed:", HUB_REPO)

In [ ]:
# Sanity check on the pushed model (streams one clip, no full re-download)
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model=HUB_REPO,
               generate_kwargs={"language": LANGUAGE, "task": "transcribe"})
strm = (load_dataset(DATASET, CONFIG, split="train", streaming=True) if CONFIG
        else load_dataset(DATASET, split="train", streaming=True))
sample = next(iter(strm))
print("PRED:", asr(sample["audio"]["array"])["text"])
print("REF :", sample[TEXT_COL])